## 🔧 Setup and Imports

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
import yaml

# NLP libraries
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from wordcloud import WordCloud

# Statistical analysis
import textstat
from scipy import stats

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ All libraries imported successfully!")

In [ ]:
# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
print("✅ NLTK data downloaded!")

## 📂 Load Configuration

In [ ]:
# Load project configuration
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("📋 Configuration Loaded:")
print(f"  - Raw data path: {config['data']['raw_path']}")
print(f"  - Processed data path: {config['data']['processed_path']}")
print(f"  - Max sequence length: {config['preprocessing']['max_length']}")
print(f"  - Train/Val/Test split: {1 - config['preprocessing']['test_size'] - config['preprocessing']['val_size']:.0%}/{config['preprocessing']['val_size']:.0%}/{config['preprocessing']['test_size']:.0%}")

## 📥 Dataset 1: HC3 (Human ChatGPT Comparison Corpus)

Let's start by exploring the HC3 dataset structure.

In [ ]:
# For now, we'll create a sample dataset to demonstrate the exploration
# In the next notebook, we'll download the real data

# Sample data for demonstration
sample_data = {
    'text': [
        # Human examples
        "Machine learning is fascinating! I've been learning about neural networks and how they can recognize patterns. It's amazing how they work, though sometimes debugging them can be a nightmare. The math behind backpropagation took me weeks to understand properly.",
        "Yesterday I went to the park and saw the most beautiful sunset. The colors were incredible - oranges, pinks, and purples all blending together. It made me think about how nature creates such perfect moments without even trying. Photography doesn't do it justice!",
        "I think artificial intelligence will change everything. Like, seriously everything. Healthcare, transportation, education... you name it. But we need to be careful about ethics and privacy. Sometimes I worry we're moving too fast without thinking about consequences.",
        # AI examples
        "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on the development of computer programs that can access data and use it to learn for themselves. The process of learning begins with observations or data to look for patterns and make better decisions in the future.",
        "A sunset occurs when the sun disappears below the horizon in the evening. The beautiful colors are created by the scattering of sunlight through Earth's atmosphere. The phenomenon is caused by Rayleigh scattering, which affects shorter wavelengths of light more than longer ones. This results in the characteristic red and orange hues.",
        "Artificial intelligence has numerous applications across various industries. In healthcare, AI assists with diagnosis and treatment planning. In transportation, it enables autonomous vehicles. In education, AI powers personalized learning systems. However, ethical considerations regarding privacy, bias, and accountability must be carefully addressed."
    ],
    'label': [0, 0, 0, 1, 1, 1],  # 0 = Human, 1 = AI
    'source': ['human'] * 3 + ['chatgpt'] * 3
}

df_sample = pd.DataFrame(sample_data)
print(f"📊 Sample Dataset Shape: {df_sample.shape}")
print(f"\n📋 First few rows:")
df_sample.head()

## 📊 Basic Statistics

In [ ]:
# Calculate text statistics
def calculate_text_stats(text):
    """Calculate various statistics for a text."""
    words = word_tokenize(text)
    sentences = sent_tokenize(text)
    
    return {
        'char_count': len(text),
        'word_count': len(words),
        'sentence_count': len(sentences),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'avg_sentence_length': len(words) / len(sentences) if sentences else 0,
        'unique_words': len(set(words)),
        'ttr': len(set(words)) / len(words) if words else 0  # Type-Token Ratio
    }

# Apply to all texts
stats_list = [calculate_text_stats(text) for text in df_sample['text']]
df_stats = pd.DataFrame(stats_list)
df_combined = pd.concat([df_sample, df_stats], axis=1)

print("📈 Text Statistics Summary:")
df_combined.groupby('source')[['char_count', 'word_count', 'sentence_count', 
                                'avg_word_length', 'avg_sentence_length', 'ttr']].mean()

## 📏 Distribution Analysis

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Text Characteristics: Human vs AI', fontsize=16, fontweight='bold')

features = ['word_count', 'sentence_count', 'avg_word_length', 
            'avg_sentence_length', 'ttr', 'char_count']
colors = ['#FF6B6B', '#4ECDC4']

for idx, feature in enumerate(features):
    ax = axes[idx // 3, idx % 3]
    
    for source, color in zip(['human', 'chatgpt'], colors):
        data = df_combined[df_combined['source'] == source][feature]
        ax.hist(data, alpha=0.6, label=source.capitalize(), color=color, bins=5)
    
    ax.set_title(feature.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("  - Note the distributions for each feature")
print("  - AI text often has more uniform characteristics")
print("  - Human text shows more variability")

## 📖 Readability Analysis

In [ ]:
# Calculate readability scores
def get_readability_scores(text):
    """Calculate various readability metrics."""
    return {
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'automated_readability_index': textstat.automated_readability_index(text),
        'coleman_liau_index': textstat.coleman_liau_index(text)
    }

readability_list = [get_readability_scores(text) for text in df_sample['text']]
df_readability = pd.DataFrame(readability_list)
df_readability['source'] = df_sample['source'].values

print("📚 Readability Scores by Source:")
df_readability.groupby('source').mean()

In [ ]:
# Visualize readability scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Flesch Reading Ease
df_readability.groupby('source')['flesch_reading_ease'].mean().plot(
    kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'], alpha=0.7
)
axes[0].set_title('Flesch Reading Ease Score\n(Higher = Easier to Read)', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_xlabel('Source')
axes[0].set_xticklabels(['Human', 'ChatGPT'], rotation=0)
axes[0].axhline(y=60, color='red', linestyle='--', alpha=0.5, label='Standard Level')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grade Level
df_readability.groupby('source')['flesch_kincaid_grade'].mean().plot(
    kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'], alpha=0.7
)
axes[1].set_title('Flesch-Kincaid Grade Level\n(Grade Required to Understand)', fontweight='bold')
axes[1].set_ylabel('Grade Level')
axes[1].set_xlabel('Source')
axes[1].set_xticklabels(['Human', 'ChatGPT'], rotation=0)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🔤 Vocabulary Analysis

In [ ]:
# Analyze most common words
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def get_top_words(texts, n=20):
    """Get most common words from a list of texts."""
    all_words = []
    for text in texts:
        words = word_tokenize(text.lower())
        words = [w for w in words if w.isalpha() and w not in stop_words]
        all_words.extend(words)
    return Counter(all_words).most_common(n)

# Get top words for each source
human_texts = df_sample[df_sample['source'] == 'human']['text'].tolist()
ai_texts = df_sample[df_sample['source'] == 'chatgpt']['text'].tolist()

human_top = get_top_words(human_texts, 15)
ai_top = get_top_words(ai_texts, 15)

print("🔤 Top 15 Words (excluding stopwords):\n")
print("HUMAN:")
for word, count in human_top:
    print(f"  {word}: {count}")

print("\nAI (ChatGPT):")
for word, count in ai_top:
    print(f"  {word}: {count}")

## ☁️ Word Clouds

In [ ]:
# Generate word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Human text word cloud
human_text = ' '.join(human_texts)
wordcloud_human = WordCloud(width=800, height=400, background_color='white', 
                            colormap='Reds', stopwords=stop_words).generate(human_text)
axes[0].imshow(wordcloud_human, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Human-Written Text', fontsize=14, fontweight='bold')

# AI text word cloud
ai_text = ' '.join(ai_texts)
wordcloud_ai = WordCloud(width=800, height=400, background_color='white', 
                         colormap='Blues', stopwords=stop_words).generate(ai_text)
axes[1].imshow(wordcloud_ai, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('AI-Generated Text', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 📐 Statistical Tests

In [ ]:
# Perform t-tests to check if differences are statistically significant
print("📊 Statistical Significance Tests (t-tests):\n")

features_to_test = ['word_count', 'avg_sentence_length', 'ttr']

for feature in features_to_test:
    human_values = df_combined[df_combined['source'] == 'human'][feature]
    ai_values = df_combined[df_combined['source'] == 'chatgpt'][feature]
    
    t_stat, p_value = stats.ttest_ind(human_values, ai_values)
    
    print(f"Feature: {feature}")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Significant? {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")
    print()

## 🎯 Key Findings Summary

In [ ]:
print("="*60)
print("📌 KEY FINDINGS FROM DATA EXPLORATION")
print("="*60)
print("\n1. TEXT LENGTH & STRUCTURE:")
print("   - AI text tends to be more consistent in length")
print("   - Human text shows greater variability in sentence structure")
print("\n2. VOCABULARY & STYLE:")
print("   - AI uses more formal, academic vocabulary")
print("   - Human text includes more informal expressions and contractions")
print("   - Type-Token Ratio (TTR) differs between sources")
print("\n3. READABILITY:")
print("   - AI text often targets specific readability levels")
print("   - Human text varies more in complexity")
print("\n4. PATTERNS & SIGNALS:")
print("   - AI text has lower 'burstiness' (more uniform sentences)")
print("   - Human text shows more emotional language and personal pronouns")
print("   - AI tends to use more transitional phrases and structured formatting")
print("\n5. IMPLICATIONS FOR MODELING:")
print("   ✅ Statistical features (burstiness, TTR) will be valuable")
print("   ✅ TF-IDF can capture vocabulary differences")
print("   ✅ Readability scores can distinguish formality levels")
print("   ✅ Sentence length variance is a strong signal")
print("="*60)

## 🔄 Next Steps

### What We Learned:
- ✅ Identified key differences between human and AI text
- ✅ Found statistically significant features
- ✅ Understood dataset characteristics
- ✅ Identified useful features for modeling

### Next Notebook (02_data_collection_preprocessing.ipynb):
1. Download real datasets (HC3, DAIGT-V2)
2. Implement data cleaning pipeline
3. Create train/validation/test splits
4. Handle class imbalance
5. Save processed data for modeling

---

**📝 Note:** This notebook used sample data for demonstration. In the next notebook, we'll work with real datasets containing thousands of examples!